# BLM Financial Assistant fine-tune (Qwen3.8-4B-Distill)

Runs the full training pipeline from the BLM MCP harness on a free Colab GPU:

1. GPU check
2. Install deps (unsloth + QLoRA stack)
3. Download train/eval JSONL from GitHub
4. Render chat template, mask prompts, cap length
5. QLoRA train (NF4 + LoRA r=16)
6. Eval holdout, smoke inference
7. Merge + GGUF Q4_K_M export
8. Optional: push to HuggingFace Hub or Google Drive

**Runtime menu: `Runtime > Change runtime type > Hardware accelerator = T4 GPU` (then Run all).**

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --upgrade transformers peft accelerate datasets bitsandbytes
!pip install -q gguf

In [ ]:
import json, urllib.request

BASE = 'https://raw.githubusercontent.com/WINDY-WINDWARD/blm-model-train/master/datasets/'
urllib.request.urlretrieve(BASE + 'train-20261105-234130.jsonl', 'train.jsonl')
urllib.request.urlretrieve(BASE + 'eval-20261105-234130.jsonl', 'eval.jsonl')

def load_jsonl(p):
    rows = []
    with open(p, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl('train.jsonl')
eval_rows = load_jsonl('eval.jsonl')
print('train', len(train_rows), 'eval', len(eval_rows))
from collections import Counter
print(Counter(r['source'] for r in train_rows))

In [ ]:
from unsloth import FastLanguageModel

max_seq_len = 4096
base = 'empero-ai/Qwen3.8-4B-Distill'
model, tokenizer = FastLanguageModel.from_pretrained(
    base,
    max_seq_length=max_seq_len,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing=True,
)
model.print_trainable_parameters()_tok = getattr(tokenizer, 'tokenizer', None)if hasattr(_tok, 'encode'):    tokenizer = _tokprint('tokenizer type:', type(tokenizer).__name__)
# --- processor unwrap fix: Qwen3.8 repo is multimodal; unsloth may return a processor ---
_tok = getattr(tokenizer, 'tokenizer', None)
if hasattr(_tok, 'encode'):
    tokenizer = _tok
print('tokenizer type:', type(tokenizer).__name__)


In [ ]:
# Mirror of train_local.py: normalize args, render template, mask prompt tokens
import math

def box(record):
    messages = record.get('messages') or []
    if not messages:
        return None
    for m in messages:
        if m.get('role') == 'assistant' and m.get('tool_calls'):
            import json as _json
            for tc in m['tool_calls']:
                fn = tc.get('function', {})
                raw = fn.get('arguments')
                if isinstance(raw, str):
                    try:
                        fn['arguments'] = _json.loads(raw) if raw.strip() else {}
                    except Exception:
                        pass
    last_idx = max(i for i, m in enumerate(messages) if m.get('role') == 'assistant')
    last = messages[last_idx]
    target_content = last.get('content') or ''
    if not target_content.strip():
        return None
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    prefix_msgs = messages[:last_idx]
    prompt_text = tokenizer.apply_chat_template(prefix_msgs, tokenize=False, add_generation_prompt=True)
    full_ids = tokenizer.encode(full_text, add_special_tokens=False)
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    if len(full_ids) > max_seq_len:
        return None
    prefix = len(prompt_ids)
    if prefix > len(full_ids):
        prefix = len(full_ids)
    labels = [-100] * prefix + full_ids[prefix:]
    return {'input_ids': full_ids, 'labels': labels}

train_feats = [f for f in (box(r) for r in train_rows) if f]
eval_feats = [f for f in (box(r) for r in eval_rows) if f]
print('train features', len(train_feats), '| eval features', len(eval_feats))
mean_len = sum(len(f['input_ids']) for f in train_feats) / max(1, len(train_feats))
print(f'mean input length: {mean_len:.0f} tokens')
from datasets import Dataset
import torch, transformers
from transformers import DataCollatorForSeq2Seq

pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
def collate(batch):
    maxlen = max(len(f['input_ids']) for f in batch)
    def pad(seq, fill):
        return seq + [fill] * (maxlen - len(seq))
    return {
        'input_ids': torch.tensor([pad(f['input_ids'], pad_id) for f in batch]),
        'labels': torch.tensor([pad(f['labels'], -100) for f in batch]),
        'attention_mask': torch.tensor([[1] * len(f['input_ids']) for f in batch]),
    }
train_ds = Dataset.from_list(train_feats)
eval_ds = Dataset.from_list(eval_feats)
print('datasets ready')

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir='out/checkpoints',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy='epoch',
    fp16=True,
    report_to=[],
    optim='paged_adamw_32bit',
    seed=42,
    dataloader_num_workers=0,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collate,
)
print('trainable tokens steps:', len(train_ds) * 2 / 8)
result = trainer.train()
print(result.metrics)
metrics = trainer.evaluate()
print('EVAL:', metrics)

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [
    {'role': 'system', 'content': 'You are a financial analyst assistant for the BLM Terminal. Use the provided tools to answer. Prefer exact tool calls over prose. When the gathered data is sufficient, stop calling tools and write the final answer. Uppercase NSE ticker symbols.'},
    {'role': 'user', 'content': 'You are an equity analyst with no access to market data tools for this task. A company reports FY25 figures in crore rupees: Revenue 120000, Operating Profit 26000, Net Profit 19000, Total Assets 180000, Shareholders Equity 90000. Calculate: (1) Operating Margin (%), (2) Net Margin (%), (3) Return on Equity (%), (4) Asset Turnover ratio. Round each answer to two decimal places.'},
]
text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))
print('---tool-call smoke---')
msgs2 = [
    {'role': 'system', 'content': 'You are a financial analyst assistant for the BLM Terminal. Use the provided tools to answer. Prefer exact tool calls over prose. When the gathered data is sufficient, stop calling tools and write the final answer. Uppercase NSE ticker symbols.'},
    {'role': 'user', 'content': 'Retrieve the order book snapshot for State Bank of India (SBIN). Calculate the bid-ask imbalance ratio, defined as (totalBuyQuantity - totalSellQuantity) / (totalBuyQuantity + totalSellQuantity). State whether buyers or sellers dominate.'},
]
text2 = tokenizer.apply_chat_template(msgs2, tokenize=False, add_generation_prompt=True)
inputs2 = tokenizer([text2], return_tensors='pt').to('cuda')
out2 = model.generate(**inputs2, max_new_tokens=256, temperature=0.1, do_sample=True)
print(tokenizer.decode(out2[0], skip_special_tokens=True))

In [ ]:
FastLanguageModel.for_inference(model)  # no-op safe
model.save_pretrained_merged('out/merged', tokenizer, save_method='merged_16bit')
model.save_pretrained_gguf('out/gguf', tokenizer, quantization_method='q4_k_m')
!ls -lh out/gguf
import shutil, pathlib
gguf_files = list(pathlib.Path('out/gguf').glob('*.gguf'))
print('GGUF files:', [str(f) for f in gguf_files])

In [ ]:
# Optional: copy to Google Drive for the llama-server swap-back on the workstation
# Give strong permissions once when prompted.
from google.colab import drive
drive.mount('/content/drive')
import shutil, pathlib
target = pathlib.Path('/content/drive/MyDrive/blm-train')
target.mkdir(parents=True, exist_ok=True)
for f in pathlib.Path('out/gguf').glob('*.gguf'):
    shutil.copy(f, target / f.name)
    print('copied to drive:', target / f.name)
print('ready: server swap-back at', target)